# CogniTrace — Fine-tune Gemma 4 E2B on Medical Communication

Fine-tunes `unsloth/gemma-4-E2B-it` with QLoRA on medical simplification pairs, exports GGUF, and pushes to HuggingFace.

**Runtime:** Kaggle T4 GPU (14.7 GB usable VRAM)  
**Dataset:** `/kaggle/input/cognitrace-medical-communication/medical_communication.jsonl` (4618 pairs)

## Cell 1 — Install Dependencies

In [ ]:
import subprocess, os
subprocess.run(["pip", "install", "unsloth"], check=True)


## Cell 2 — Configuration

All hyperparameters in one place. `HF_TOKEN` is read from Kaggle Secrets (set in notebook settings under Add-ons > Secrets).

In [ ]:
import os

# Model
MODEL_ID = "unsloth/gemma-4-E2B-it"
MAX_SEQ_LENGTH = 2048
LOAD_IN_4BIT = True

# LoRA
LORA_R = 16
LORA_ALPHA = 16
LORA_DROPOUT = 0
LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

# Training
NUM_EPOCHS = 3
LEARNING_RATE = 2e-4
BATCH_SIZE = 2
GRAD_ACCUM_STEPS = 4
WARMUP_STEPS = 5
WEIGHT_DECAY = 0.01
OPTIMIZER = "adamw_8bit"
LR_SCHEDULER = "linear"
SEED = 3407

# Paths
DATA_PATH = "/kaggle/input/cognitrace-medical-communication/medical_communication.jsonl"
if not os.path.exists(DATA_PATH):
    import subprocess
    subprocess.run(["pip", "install", "-q", "kagglehub"], check=True)
    import kagglehub
    dataset_slug = os.environ.get("KAGGLE_DATASET_SLUG", "INSERT_USERNAME/cognitrace-medical-communication")
    dl_path = kagglehub.dataset_download(dataset_slug)
    DATA_PATH = os.path.join(dl_path, "medical_communication.jsonl")
    print(f"Dataset downloaded to: {DATA_PATH}")
OUTPUT_DIR = "/kaggle/working/lora_model"
GGUF_DIR = "/kaggle/working/gguf"

# HuggingFace
HF_REPO = "littlebull9/cognitrace-gemma4-medical-GGUF"
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN", "")

print(f"Model:      {MODEL_ID}")
print(f"LoRA r:     {LORA_R}")
print(f"Epochs:     {NUM_EPOCHS}")
print(f"Batch size: {BATCH_SIZE} (effective: {BATCH_SIZE * GRAD_ACCUM_STEPS})")
print(f"HF_TOKEN:   {'set' if HF_TOKEN else 'NOT SET -- will save locally only'}")


## Cell 3 — Load Model

Loads Gemma 4 E2B with 4-bit quantization via Unsloth. Expected VRAM usage is ~8 GB on T4.

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=LOAD_IN_4BIT,
)

gpu_stats = torch.cuda.get_device_properties(0)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")

## Cell 4 — Add LoRA Adapters

Attaches QLoRA adapters to all attention and MLP projection layers. `use_gradient_checkpointing="unsloth"` resolves the `use_cache` conflict during training.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=LORA_TARGET_MODULES,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
    use_rslora=False,
    loftq_config=None,
)

## Cell 5 — Load and Format Dataset

Reads the JSONL file from the Kaggle dataset input. Each example's `messages` list is formatted with the model's chat template.

In [ ]:
from datasets import load_dataset

raw = load_dataset("json", data_files=DATA_PATH, split="train")

def format_example(example):
    messages = example["messages"]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return {"text": text}

dataset = raw.map(format_example, num_proc=1)
print(f"Loaded {len(dataset)} training pairs")
print(f"Sample:\n{dataset[0]['text'][:500]}")

## Cell 6 — Train

SFTTrainer with `packing=False` for clean per-example supervision. Uses bf16 on T4 if supported, otherwise fp16. Target final loss is under 1.0.

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=SFTConfig(
        dataset_text_field="text",
        max_seq_length=MAX_SEQ_LENGTH,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM_STEPS,
        warmup_steps=WARMUP_STEPS,
        num_train_epochs=NUM_EPOCHS,
        learning_rate=LEARNING_RATE,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=10,
        optim=OPTIMIZER,
        weight_decay=WEIGHT_DECAY,
        lr_scheduler_type=LR_SCHEDULER,
        seed=SEED,
        output_dir=OUTPUT_DIR,
        report_to="none",
        packing=False,
    ),
)

print("Starting training...")
stats = trainer.train()
print(f"Final loss: {stats.training_loss:.4f}")
print(f"Total steps: {stats.global_step}")


## Cell 7 — Save LoRA Adapter

Saves the trained adapter weights and tokenizer to Kaggle working directory.

In [ ]:
import os, shutil

# Clean disk before saving
for d in ["/kaggle/working/unsloth_compiled_cache", "/root/.cache/pip"]:
    if os.path.exists(d):
        shutil.rmtree(d, ignore_errors=True)
        print(f"Cleared {d}")

os.makedirs(OUTPUT_DIR, exist_ok=True)
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

adapter_size = sum(
    os.path.getsize(os.path.join(OUTPUT_DIR, f))
    for f in os.listdir(OUTPUT_DIR)
    if os.path.isfile(os.path.join(OUTPUT_DIR, f))
) / (1024 * 1024)
print(f"LoRA adapter saved to {OUTPUT_DIR} ({adapter_size:.0f} MB)")

if HF_TOKEN:
    model.push_to_hub(HF_REPO, token=HF_TOKEN)
    tokenizer.push_to_hub(HF_REPO, token=HF_TOKEN)
    print(f"LoRA adapter pushed to: https://huggingface.co/{HF_REPO}")


## Cell 8 — Export GGUF

Merges the LoRA adapter and exports Q4_K_M quantized GGUF for on-device inference (CoreML/llama.cpp).

In [ ]:
# GGUF export skipped on Kaggle (disk space limit).
# Merge + GGUF export will be done locally with run_eval_local.sh
print("LoRA adapter saved. GGUF export will be done locally (more disk space).")
gguf_path = "pending local export"
size_mb = 0


## Cell 9 — Push to HuggingFace

Uploads the GGUF to the HuggingFace Hub. Requires `HF_TOKEN` in Kaggle Secrets (Add-ons > Secrets > HF_TOKEN). If not set, the GGUF is available in the Kaggle Output tab.

In [ ]:
# HF push already handled in GGUF export cell above.
print("GGUF export and HF push complete.")


## Cell 10 — Sanity Check

Quick inference test to confirm the fine-tuned model generates coherent medical simplifications.

In [ ]:
FastLanguageModel.for_inference(model)

messages = [{"role": "user", "content": "Rewrite this for a patient: The patient presents with bradykinesia and resting tremor consistent with early-stage Parkinson's disease."}]
inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
outputs = model.generate(input_ids=inputs, max_new_tokens=256, use_cache=True, temperature=0.7)
response = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True)
print(f"Test response:\n{response}")


## Cell 11 — Training Summary

In [ ]:
print("=" * 60)
print("TRAINING COMPLETE")
print("=" * 60)
print(f"Model: {MODEL_ID}")
print(f"Dataset: {len(dataset)} pairs")
print(f"Epochs: {NUM_EPOCHS}")
print(f"Final loss: {stats.training_loss:.4f}")
print(f"LoRA adapter: {OUTPUT_DIR}")
if HF_TOKEN:
    print(f"HuggingFace: https://huggingface.co/{HF_REPO}")
print(f"Next step: merge + GGUF export locally")
print("=" * 60)
